In [ ]:
import re
import time
import math
import random
from typing import List, Dict, Any
import pandas as pd
import yfinance as yf
from astrapy import DataAPIClient
from groq import Groq

from concurrent.futures import ThreadPoolExecutor, as_completed, wait
from threading import BoundedSemaphore

from confluent_kafka import Consumer, KafkaError, KafkaException
from pathlib import Path
import json

from pyspark.sql import SparkSession
from pyspark.sql.types import *

import time
from datetime import datetime, timezone
from typing import List, Optional
import threading

from pyspark.sql.types import StructType, StructField, StringType, IntegerType, LongType, TimestampType

In [ ]:
'''
    This notebook contains code snippets for consuming data from various streaming sources.
    It demonstrates how to set up clients for different data providers and manage concurrent data processing.
    For calculating the sentiment score, a LLM is called via Groq API.
    The data is finally stored in Astra DB (based on Apache Cassandra) using AstraPy client.

    Due to restriction on LLM calls in Groq free tier, the code related to sentiment score calculation is commented out.
    The sentiment is pre-calculated and stored in the dataset used for testing the code.
'''

In [ ]:

grok_api_key = "YOUR_GROQ_API_KEY_HERE"
db_end_point = 'https://d1091311-1ba8-4970-bd74-937d6cda55b0-us-east-2.apps.astra.datastax.com'
db_token = 'AstraCS:YOUR_TOKEN_HERE'

In [ ]:
#Python script to consume Kafka messages from multiple topics using one consumer per topic,
#normalize the data into a Spark DataFrame, and insert into Astra DB using AstraPy.

import json
import time
from pathlib import Path
from datetime import datetime, timezone
from typing import List, Optional
import threading
from pyspark.sql import SparkSession
from pyspark.sql.types import *
import json
import pandas as pd
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, LongType, TimestampType

import pandas as pd
from confluent_kafka import Consumer, KafkaError, KafkaException
from astrapy import DataAPIClient

# ----- CONFIG -----
BOOTSTRAP = "localhost:9092"
TOPICS = [
    'NASDAQ_AMGN'
]
GROUP = "one-shot-debug-group"   # base name for groups; we append topic to make unique groups
INACTIVITY_TIMEOUT_SECONDS = 5   # stop a consumer if no messages for this many seconds
SAVE_CSV = False                 # set True to write out/one_shot.csv
OUT = Path("out/one_shot.csv")

# NOTE: conf will be cloned per consumer and group id adjusted
base_conf = {
    "bootstrap.servers": BOOTSTRAP,
    "auto.offset.reset": "earliest",
    "enable.auto.commit": False
}

def parse_value(b) -> Optional[object]:
    """Decode message value: bytes -> JSON dict (if possible) or raw string."""
    if b is None:
        return None
    try:
        s = b.decode("utf-8") if isinstance(b, (bytes, bytearray)) else str(b)
        return json.loads(s)
    except Exception:
        try:
            return b.decode("utf-8")
        except Exception:
            return str(b)

def normalize_record(msg):
    """Return normalized dict for DataFrame."""
    value = parse_value(msg.value())

    stock_symbol = None
    summary = None
    published_date = None
    url = None
    sentiment_score = None

    if isinstance(value, dict):
        stock_symbol = value.get("stock_symbol") or value.get("symbol") or value.get("ticker")
        summary = value.get("summary") or value.get("text") or value.get("headline")
        published_date = value.get("event_time") or value.get("published_date") or value.get("timestamp")
        url = value.get("url") or value.get("link")
        sentiment_score = value.get("sentiment_score")
    else:
        # non-dict payload: keep raw value in summary for inspection
        summary = str(value)

    # Try to produce a timezone-aware datetime for published_date
    published_ts = None
    if published_date is not None:
        try:
            published_ts = pd.to_datetime(published_date, utc=True)
        except Exception:
            try:
                # maybe epoch millis
                if isinstance(published_date, (int, float)) or str(published_date).isdigit():
                    published_ts = datetime.fromtimestamp(int(published_date) / 1000.0, tz=timezone.utc)
            except Exception:
                published_ts = None

    # kafka timestamp (type: (type, timestamp_ms))
    kafka_ts = None
    try:
        kafka_ts = msg.timestamp()[1]
    except Exception:
        kafka_ts = None

    key = msg.key()
    if isinstance(key, (bytes, bytearray)):
        try:
            key = key.decode("utf-8")
        except Exception:
            key = str(key)

    rec = {
        "topic": msg.topic(),
        "partition": msg.partition(),
        "offset": msg.offset(),
        "kafka_timestamp_ms": kafka_ts,
        "kafka_received_at": datetime.now(timezone.utc).isoformat(),
        "key": key,
        "value_raw": value,
        "stock_symbol": stock_symbol,
        "summary": summary,
        "published_date_raw": published_date,
        "published_ts": published_ts,
        "url": url,
        "sentiment_score": sentiment_score,
    }
    return rec

# ----- PER-TOPIC CONSUMER THREAD -----
def _consume_topic_thread(topic: str,
                          results_list: List[List[dict]],
                          results_lock: threading.Lock,
                          inactivity_timeout: int = INACTIVITY_TIMEOUT_SECONDS):
    """
    Consume messages for a single topic until inactivity_timeout seconds pass without messages.
    Append the list of records for this topic to results_list (thread-safe).
    """
    # make a consumer conf unique for this topic (unique group so each consumer reads independently)
    conf = dict(base_conf)  
    # create a unique group id per topic run so each consumer can read earliest
    conf["group.id"] = f"{GROUP}-{topic}-{int(time.time()*1000)}"

    c = Consumer(conf)
    c.subscribe([topic])
    print(f"[{topic}] Subscribed (group={conf['group.id']})")

    records = []
    start = time.time()
    last_msg_time = time.time()

    try:
        while True:
            # stop if we've been inactive for the timeout period and we already received records
            if (time.time() - last_msg_time) > inactivity_timeout and records:
                print(f"[{topic}] No messages for {inactivity_timeout}s and already received records -> stopping.")
                break
            # guard: stop eventually if nothing ever arrives (prevent infinite loop)
            if (time.time() - start) > (inactivity_timeout * 20) and not records:
                print(f"[{topic}] No messages arrived within maximum wait window -> stopping.")
                break

            msg = c.poll(timeout=1.0)
            if msg is None:
                continue
            if msg.error():
                if msg.error().code() == KafkaError._PARTITION_EOF:
                    # partition EOF is not fatal; continue
                    continue
                print(f"[{topic}] Consumer error:", msg.error())
                continue

            rec = normalize_record(msg)
            print(f"[{topic}] Got message: partition {rec['partition']} offset {rec['offset']} symbol {rec['stock_symbol']} sentiment {rec['sentiment_score']}")
            records.append(rec)
            last_msg_time = time.time()

    except KeyboardInterrupt:
        print(f"[{topic}] Interrupted by user.")
    except KafkaException as ke:
        print(f"[{topic}] Kafka exception:", ke)
    finally:
        try:
            c.close()
        except Exception:
            pass
        print(f"[{topic}] Consumer closed. Collected {len(records)} records.")

    # append the result atomically
    with results_lock:
        results_list.append(records)

def consume_one_consumer_per_topic(topics=TOPICS, inactivity_timeout=INACTIVITY_TIMEOUT_SECONDS):
    """
    Spawn one thread/consumer per topic and return a Spark DataFrame of all records.
    """
    threads = []
    results = []  # will hold lists of records, one per topic  
    lock = threading.Lock()
    print("Starting consumers...")

    for t in topics:
        thr = threading.Thread(target=_consume_topic_thread,
                               args=(t, results, lock, inactivity_timeout),
                               daemon=True)
        thr.start()
        threads.append(thr)

    for thr in threads:
        thr.join()

    # Flatten results
    all_records = []
    for part in results:
        if part:
            all_records.extend(part)

    if not all_records:
        print("No records received from any topic.")
        return spark.createDataFrame([], schema="topic STRING, partition INT, offset LONG")

    print(f"\nTotal records (all topics): {len(all_records)}")

    def sanitize_record(r: dict) -> dict:
        # topic
        topic = str(r.get("topic")) if r.get("topic") is not None else None

        # partition/offset -> int if possible else None
        try:
            partition = int(r.get("partition")) if r.get("partition") is not None else None
        except Exception:
            partition = None
        try:
            offset = int(r.get("offset")) if r.get("offset") is not None else None
        except Exception:
            offset = None

        # kafka_timestamp_ms -> int or None
        try:
            kafka_timestamp_ms = int(r.get("kafka_timestamp_ms")) if r.get("kafka_timestamp_ms") is not None else None
        except Exception:
            kafka_timestamp_ms = None

        # kafka_received_at -> keep ISO string
        kafka_received_at = r.get("kafka_received_at")
        if kafka_received_at is not None:
            kafka_received_at = str(kafka_received_at)

        # key -> str or None
        key = r.get("key")
        key = str(key) if key is not None else None

        # value_raw -> JSON string (so Spark sees a string, not an arbitrary dict)
        val = r.get("value_raw")
        try:
            if isinstance(val, (dict, list)):
                value_raw = json.dumps(val)
            else:
                value_raw = str(val) if val is not None else None
        except Exception:
            value_raw = str(val)

        # stock_symbol, summary, url -> strings (or None)
        stock_symbol = str(r.get("stock_symbol")) if r.get("stock_symbol") is not None else None
        summary = str(r.get("summary")) if r.get("summary") is not None else None
        url = str(r.get("url")) if r.get("url") is not None else None

        # published_date_raw -> string or None
        published_date_raw = r.get("published_date_raw")
        published_date_raw = str(published_date_raw) if published_date_raw is not None else None

        # published_ts -> normalize to Python datetime (or None)
        published_ts = r.get("published_ts")
        try:
            if isinstance(published_ts, pd.Timestamp):
                published_ts = published_ts.to_pydatetime()
            elif isinstance(published_ts, str):
                # pandas will handle many ISO/offset formats
                published_ts = pd.to_datetime(published_ts, utc=True, errors="coerce")
                if pd.isna(published_ts):
                    published_ts = None
                else:
                    published_ts = published_ts.to_pydatetime()
            elif isinstance(published_ts, (int, float)):
                # epoch millis?
                published_ts = datetime.fromtimestamp(int(published_ts) / 1000.0, tz=timezone.utc)
            # else leave as-is (if datetime)
            if published_ts is not None and not isinstance(published_ts, datetime):
                # last resort: try to coerce via pandas
                tmp = pd.to_datetime(published_ts, utc=True, errors="coerce")
                published_ts = tmp.to_pydatetime() if not pd.isna(tmp) else None
        except Exception:
            published_ts = None

        # sentiment_score -> int or None
        sentiment_score = r.get("sentiment_score")
        try:
            sentiment_score = int(sentiment_score) if sentiment_score is not None else None
        except Exception:
            sentiment_score = None

        return {
            "topic": topic,
            "partition": partition,
            "offset": offset,
            "kafka_timestamp_ms": kafka_timestamp_ms,
            "kafka_received_at": kafka_received_at,
            "key": key,
            "value_raw": value_raw,
            "stock_symbol": stock_symbol,
            "summary": summary,
            "published_date_raw": published_date_raw,
            "published_ts": published_ts,
            "url": url,
            "sentiment_score": sentiment_score,
        }

    sanitized = [sanitize_record(r) for r in all_records]

    # define explicit schema matching sanitized record types
    schema = StructType([
        StructField("topic", StringType(), True),
        StructField("partition", IntegerType(), True),
        StructField("offset", LongType(), True),
        StructField("kafka_timestamp_ms", LongType(), True),
        StructField("kafka_received_at", StringType(), True),
        StructField("key", StringType(), True),
        StructField("value_raw", StringType(), True),
        StructField("stock_symbol", StringType(), True),
        StructField("summary", StringType(), True),
        StructField("published_date_raw", StringType(), True),
        StructField("published_ts", TimestampType(), True),
        StructField("url", StringType(), True),
        StructField("sentiment_score", IntegerType(), True),
    ])

    # create DataFrame with explicit schema
    spark_df = spark.createDataFrame(sanitized, schema=schema)

    print("Created Spark DataFrame with schema:")
    spark_df.printSchema()

    return spark_df

#handle date formats
def _to_utc_iso(dt):
    """
    Convert various datetime-like inputs to an ISO8601 string with UTC tz,
    or return None if input is missing/invalid.
    """
    if dt is None or (isinstance(dt, float) and math.isnan(dt)):
        return None
    try:
        # Let pandas normalize many inputs (str, pd.Timestamp, datetime, numpy types)
        ts = pd.to_datetime(dt, utc=True, errors="coerce")
        if pd.isna(ts):
            return None
        # ts is tz-aware (UTC). Return ISO string with offset Z (e.g. 2025-11-27T08:00:00+00:00)
        return ts.isoformat()
    except Exception:
        return None

def insert_data_to_db_spark(spark_df, db_end_point: str, db_token: str, chunk_size: int = 500):
    """
    Insert rows from a Spark DataFrame into Astra DB (DataAPI).
    Ensures published_at is timezone-aware ISO strings (UTC) to avoid Serdes errors.
    Uses chunked insert_many to avoid very large single requests.

    Args:
        spark_df: Spark DataFrame containing at least these columns:
                  stock_symbol, summary, published_ts, url, sentiment_score
        db_end_point: Astra DB endpoint URL
        db_token: Astra DB token
        chunk_size: number of rows per insert_many call
    """
    cols = []
    for c in ("stock_symbol", "summary", "published_ts", "url", "sentiment_score"):
        if c in spark_df.columns:
            cols.append(c)
        else:
            spark_df = spark_df.withColumn(c, spark_df[c] * 0) if False else spark_df 

    pdf = spark_df.select("stock_symbol", "summary", "published_ts", "url", "sentiment_score").toPandas()

    if pdf.empty:
        print("No records to insert.")
        return

    # Build rows while normalizing published_at to tz-aware ISO strings
    rows = []
    for idx in range(len(pdf)):
        symbol = pdf.loc[idx, "stock_symbol"] if "stock_symbol" in pdf.columns else None
        summary = pdf.loc[idx, "summary"] if "summary" in pdf.columns else None
        pub_date = pdf.loc[idx, "published_ts"] if "published_ts" in pdf.columns else None
        news_url = pdf.loc[idx, "url"] if "url" in pdf.columns else None
        sentiment_score = pdf.loc[idx, "sentiment_score"] if "sentiment_score" in pdf.columns else None

        # convert published_ts to UTC ISO string (or None)
        published_at_iso = _to_utc_iso(pub_date)

        # Safe cast sentiment
        try:
            sentiment_score_int = int(sentiment_score) if sentiment_score is not None and not (isinstance(sentiment_score, float) and math.isnan(sentiment_score)) else None
        except Exception:
            sentiment_score_int = None

        row = {
            "stock_symbol": None if pd.isna(symbol) else symbol,
            "published_at": published_at_iso,
            "news_url": None if pd.isna(news_url) else news_url,
            "news_summary": None if pd.isna(summary) else summary,
            "sentiment_score": sentiment_score_int
        }
        rows.append(row)

    # init client and table
    client_db = DataAPIClient()
    database = client_db.get_database(db_end_point, token=db_token)
    table = database.get_table("news_sentiment")

    print(f"Prepared {len(rows)} rows. Inserting in chunks of {chunk_size}...")

    for i in range(0, len(rows), chunk_size):
        chunk = rows[i:i + chunk_size]
        try:
            table.insert_many(chunk)
            print(f"Inserted rows {i}..{i+len(chunk)-1}")
        except Exception as e:
            print(f"Failed to insert chunk starting at {i}: {e}")
            # optionally: raise or continue
            raise

    print("All done.")

# ----- MAIN -----
if __name__ == "__main__":
    spark = SparkSession.builder.appName("KafkaConsumerToSpark").getOrCreate()
    df = consume_one_consumer_per_topic()
        
    db_end_point = 'https://d1091311-1ba8-4970-bd74-937d6cda55b0-us-east-2.apps.astra.datastax.com'
    db_token = 'AstraCS:YOUR_TOKEN_HERE'
    insert_data_to_db_spark(df, db_end_point, db_token)


In [ ]:
#Python script to consume Kafka messages from multiple topics using one consumer per topic with code for calculating performance metrics.
import json
import time
from pathlib import Path
from datetime import datetime, timezone
from typing import List, Optional
import threading
from pyspark.sql import SparkSession
from pyspark.sql.types import *
import json
import pandas as pd
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, LongType, TimestampType
import math
import numpy as np
from astrapy import DataAPIClient

import pandas as pd
from confluent_kafka import Consumer, KafkaError, KafkaException

# ----- CONFIG -----
BOOTSTRAP = "localhost:9092"
TOPICS = ['NASDAQ_AMGN','NASDAQ_TSM','NASDAQ_CMG','NASDAQ_ORCL','NASDAQ_GE','NASDAQ_CMCSA','NASDAQ_PYPL','NASDAQ_EBAY','NASDAQ_BIIB','NASDAQ_QCOM','NASDAQ_AMD',
 'NASDAQ_COST','NASDAQ_CRM','NASDAQ_BABA','NASDAQ_COP','NASDAQ_CVX','NASDAQ_USO','NASDAQ_NKE','NASDAQ_WFC','NASDAQ_MRK','NASDAQ_AAL','NASDAQ_GSK','NASDAQ_QQQ',
 'NASDAQ_PEP','NASDAQ_ABBV']

GROUP = "one-shot-debug-group"   # base name for groups; we append topic to make unique groups
INACTIVITY_TIMEOUT_SECONDS = 5   # stop a consumer if no messages for this many seconds
SAVE_CSV = False                 # set True to write out/one_shot.csv
OUT = Path("out/one_shot.csv")
SAVE_METRICS_JSON = False        # set True to write metrics to out/metrics.json
METRICS_OUT = Path("out/metrics.json")

base_conf = {
    "bootstrap.servers": BOOTSTRAP,
    "auto.offset.reset": "earliest",
    "enable.auto.commit": False
}

# ----- HELPERS -----
def parse_value(b) -> Optional[object]:
    """Decode message value: bytes -> JSON dict (if possible) or raw string."""
    if b is None:
        return None
    try:
        s = b.decode("utf-8") if isinstance(b, (bytes, bytearray)) else str(b)
        return json.loads(s)
    except Exception:
        try:
            return b.decode("utf-8")
        except Exception:
            return str(b)

def normalize_record(msg):
    """Return normalized dict for DataFrame."""
    value = parse_value(msg.value())

    stock_symbol = None
    summary = None
    published_date = None
    url = None
    sentiment_score = None

    if isinstance(value, dict):
        stock_symbol = value.get("stock_symbol") or value.get("symbol") or value.get("ticker")
        summary = value.get("summary") or value.get("text") or value.get("headline")
        published_date = value.get("event_time") or value.get("published_date") or value.get("timestamp")
        url = value.get("url") or value.get("link")
        sentiment_score = value.get("sentiment_score")
    else:
        summary = str(value)

    published_ts = None
    if published_date is not None:
        try:
            published_ts = pd.to_datetime(published_date, utc=True)
        except Exception:
            try:
                # maybe epoch millis
                if isinstance(published_date, (int, float)) or str(published_date).isdigit():
                    published_ts = datetime.fromtimestamp(int(published_date) / 1000.0, tz=timezone.utc)
            except Exception:
                published_ts = None

    # kafka timestamp (type: (type, timestamp_ms))
    kafka_ts = None
    try:
        kafka_ts = msg.timestamp()[1]
    except Exception:
        kafka_ts = None

    key = msg.key()
    if isinstance(key, (bytes, bytearray)):
        try:
            key = key.decode("utf-8")
        except Exception:
            key = str(key)

    rec = {
        "topic": msg.topic(),
        "partition": msg.partition(),
        "offset": msg.offset(),
        "kafka_timestamp_ms": kafka_ts,
        "kafka_received_at": datetime.now(timezone.utc).isoformat(),
        "key": key,
        "value_raw": value,
        "stock_symbol": stock_symbol,
        "summary": summary,
        "published_date_raw": published_date,
        "published_ts": published_ts,
        "url": url,
        "sentiment_score": sentiment_score,
    }
    return rec

# ----- PER-TOPIC CONSUMER THREAD -----
def _consume_topic_thread(topic: str,
                          results_list: List[List[dict]],
                          results_lock: threading.Lock,
                          inactivity_timeout: int = INACTIVITY_TIMEOUT_SECONDS):
    """
    Consume messages for a single topic until inactivity_timeout seconds pass without messages.
    Append the list of records for this topic to results_list (thread-safe).
    """
    # make a consumer conf unique for this topic (unique group so each consumer reads independently)
    conf = dict(base_conf)  # shallow copy
    # create a unique group id per topic run so each consumer can read earliest
    conf["group.id"] = f"{GROUP}-{topic}-{int(time.time()*1000)}"

    c = Consumer(conf)
    c.subscribe([topic])
    print(f"[{topic}] Subscribed (group={conf['group.id']})")

    records = []
    start = time.time()
    last_msg_time = time.time()

    try:
        while True:
            # stop if we've been inactive for the timeout period and we already received records
            if (time.time() - last_msg_time) > inactivity_timeout and records:
                print(f"[{topic}] No messages for {inactivity_timeout}s and already received records -> stopping.")
                break
            # guard: stop eventually if nothing ever arrives (prevent infinite loop)
            if (time.time() - start) > (inactivity_timeout * 20) and not records:
                print(f"[{topic}] No messages arrived within maximum wait window -> stopping.")
                break

            msg = c.poll(timeout=1.0)
            if msg is None:
                continue
            if msg.error():
                if msg.error().code() == KafkaError._PARTITION_EOF:
                    # partition EOF is not fatal; continue
                    continue
                print(f"[{topic}] Consumer error:", msg.error())
                continue

            rec = normalize_record(msg)
            # debug line
            print(f"[{topic}] Got message: partition {rec['partition']} offset {rec['offset']} symbol {rec['stock_symbol']} sentiment {rec['sentiment_score']}")
            records.append(rec)
            last_msg_time = time.time()

    except KeyboardInterrupt:
        print(f"[{topic}] Interrupted by user.")
    except KafkaException as ke:
        print(f"[{topic}] Kafka exception:", ke)
    finally:
        try:
            c.close()
        except Exception:
            pass
        print(f"[{topic}] Consumer closed. Collected {len(records)} records.")

    # append the result atomically
    with results_lock:
        results_list.append(records)

def consume_one_consumer_per_topic(topics=TOPICS, inactivity_timeout=INACTIVITY_TIMEOUT_SECONDS):
    """
    Spawn one thread/consumer per topic and return a Spark DataFrame of all records.
    """
    threads = []
    results = []  # will hold lists of records, one per topic  
    lock = threading.Lock()
    print("Starting consumers...")

    for t in topics:
        thr = threading.Thread(target=_consume_topic_thread,
                               args=(t, results, lock, inactivity_timeout),
                               daemon=True)
        thr.start()
        threads.append(thr)

    for thr in threads:
        thr.join()

    # Flatten results
    all_records = []
    for part in results:
        if part:
            all_records.extend(part)

    if not all_records:
        print("No records received from any topic.")
        # return empty spark df with minimal schema
        return spark.createDataFrame([], schema="topic STRING, partition INT, offset LONG")

    print(f"\nTotal records (all topics): {len(all_records)}")

    def sanitize_record(r: dict) -> dict:
        # topic
        topic = str(r.get("topic")) if r.get("topic") is not None else None

        # partition/offset -> int if possible else None
        try:
            partition = int(r.get("partition")) if r.get("partition") is not None else None
        except Exception:
            partition = None
        try:
            offset = int(r.get("offset")) if r.get("offset") is not None else None
        except Exception:
            offset = None

        # kafka_timestamp_ms -> int or None
        try:
            kafka_timestamp_ms = int(r.get("kafka_timestamp_ms")) if r.get("kafka_timestamp_ms") is not None else None
        except Exception:
            kafka_timestamp_ms = None

        # kafka_received_at -> keep ISO string
        kafka_received_at = r.get("kafka_received_at")
        if kafka_received_at is not None:
            kafka_received_at = str(kafka_received_at)

        # key -> str or None
        key = r.get("key")
        key = str(key) if key is not None else None

        # value_raw -> JSON string (so Spark sees a string, not an arbitrary dict)
        val = r.get("value_raw")
        try:
            if isinstance(val, (dict, list)):
                value_raw = json.dumps(val)
            else:
                value_raw = str(val) if val is not None else None
        except Exception:
            value_raw = str(val)

        # stock_symbol, summary, url -> strings (or None)
        stock_symbol = str(r.get("stock_symbol")) if r.get("stock_symbol") is not None else None
        summary = str(r.get("summary")) if r.get("summary") is not None else None
        url = str(r.get("url")) if r.get("url") is not None else None

        # published_date_raw -> string or None
        published_date_raw = r.get("published_date_raw")
        published_date_raw = str(published_date_raw) if published_date_raw is not None else None

        # published_ts -> normalize to Python datetime (or None)
        published_ts = r.get("published_ts")
        try:
            if isinstance(published_ts, pd.Timestamp):
                published_ts = published_ts.to_pydatetime()
            elif isinstance(published_ts, str):
                # pandas will handle many ISO/offset formats
                published_ts = pd.to_datetime(published_ts, utc=True, errors="coerce")
                if pd.isna(published_ts):
                    published_ts = None
                else:
                    published_ts = published_ts.to_pydatetime()
            elif isinstance(published_ts, (int, float)):
                # epoch millis?
                published_ts = datetime.fromtimestamp(int(published_ts) / 1000.0, tz=timezone.utc)
            # else leave as-is (if datetime)
            if published_ts is not None and not isinstance(published_ts, datetime):
                # last resort: try to coerce via pandas
                tmp = pd.to_datetime(published_ts, utc=True, errors="coerce")
                published_ts = tmp.to_pydatetime() if not pd.isna(tmp) else None
        except Exception:
            published_ts = None

        # sentiment_score -> int or None
        sentiment_score = r.get("sentiment_score")
        try:
            sentiment_score = int(sentiment_score) if sentiment_score is not None else None
        except Exception:
            sentiment_score = None

        return {
            "topic": topic,
            "partition": partition,
            "offset": offset,
            "kafka_timestamp_ms": kafka_timestamp_ms,
            "kafka_received_at": kafka_received_at,
            "key": key,
            "value_raw": value_raw,
            "stock_symbol": stock_symbol,
            "summary": summary,
            "published_date_raw": published_date_raw,
            "published_ts": published_ts,
            "url": url,
            "sentiment_score": sentiment_score,
        }

    sanitized = [sanitize_record(r) for r in all_records]

    # define explicit schema matching sanitized record types
    schema = StructType([
        StructField("topic", StringType(), True),
        StructField("partition", IntegerType(), True),
        StructField("offset", LongType(), True),
        StructField("kafka_timestamp_ms", LongType(), True),
        StructField("kafka_received_at", StringType(), True),
        StructField("key", StringType(), True),
        StructField("value_raw", StringType(), True),
        StructField("stock_symbol", StringType(), True),
        StructField("summary", StringType(), True),
        StructField("published_date_raw", StringType(), True),
        StructField("published_ts", TimestampType(), True),
        StructField("url", StringType(), True),
        StructField("sentiment_score", IntegerType(), True),
    ])

    # create DataFrame with explicit schema
    spark_df = spark.createDataFrame(sanitized, schema=schema)

    print("Created Spark DataFrame with schema:")
    spark_df.printSchema()

    return spark_df

# ----- METRICS -----
def _ms_from_iso(s):
    """Safe conversion from ISO string to pandas Timestamp (UTC)"""
    try:
        return pd.to_datetime(s, utc=True)
    except Exception:
        return pd.NaT

def compute_pipeline_metrics(spark_df, consumption_elapsed_seconds: float = None):
    """
    Compute and print performance metrics for the pipeline based on the Spark DataFrame.
    Returns a dictionary with metrics.
    """
    # convert to pandas for easy analysis
    pdf = spark_df.toPandas()
    if pdf.empty:
        print("No data for metrics.")
        return {}

    pdf["kafka_ts_dt"] = pd.to_datetime(pdf["kafka_timestamp_ms"], unit="ms", utc=True, errors="coerce")
    pdf["kafka_received_at_dt"] = pd.to_datetime(pdf["kafka_received_at"], utc=True, errors="coerce")
    pdf["published_ts_dt"] = pd.to_datetime(pdf["published_ts"], utc=True, errors="coerce")

    # message size approx (bytes) based on value_raw string
    def _size_bytes(x):
        try:
            return len(x.encode("utf-8"))
        except Exception:
            try:
                return len(str(x))
            except Exception:
                return 0
    pdf["msg_size_bytes"] = pdf["value_raw"].apply(lambda x: _size_bytes(x if x is not None else ""))

    # latency measures
    # kafka -> received
    pdf["kafka_to_received_ms"] = (pdf["kafka_received_at_dt"] - pdf["kafka_ts_dt"]).dt.total_seconds() * 1000.0
    # published -> received (end-to-end) when published_ts present
    pdf["published_to_received_ms"] = (pdf["kafka_received_at_dt"] - pdf["published_ts_dt"]).dt.total_seconds() * 1000.0

    # basic counts and throughput
    total_msgs = len(pdf)
    elapsed = consumption_elapsed_seconds if consumption_elapsed_seconds is not None else None
    throughput = (total_msgs / elapsed) if (elapsed and elapsed > 0) else None

    # per-topic metrics
    topic_groups = pdf.groupby("topic")
    per_topic = {}
    for t, g in topic_groups:
        per_topic[t] = {
            "count": int(len(g)),
            "avg_msg_size_bytes": float(g["msg_size_bytes"].mean()) if len(g) > 0 else None,
            "min_msg_size_bytes": int(g["msg_size_bytes"].min()) if len(g) > 0 else None,
            "max_msg_size_bytes": int(g["msg_size_bytes"].max()) if len(g) > 0 else None,
        }

    # overall message size stats
    size_stats = {
        "count": int(total_msgs),
        "mean_bytes": float(pdf["msg_size_bytes"].mean()),
        "p50_bytes": int(pdf["msg_size_bytes"].quantile(0.5)),
        "p90_bytes": int(pdf["msg_size_bytes"].quantile(0.9)),
        "p95_bytes": int(pdf["msg_size_bytes"].quantile(0.95)),
        "max_bytes": int(pdf["msg_size_bytes"].max()),
    }

    # latency stats helper
    def _lat_stats(series):
        s = series.dropna()
        if s.empty:
            return {}
        return {
            "count": int(len(s)),
            "mean_ms": float(s.mean()),
            "median_ms": float(s.quantile(0.5)),
            "p90_ms": float(s.quantile(0.9)),
            "p95_ms": float(s.quantile(0.95)),
            "p99_ms": float(s.quantile(0.99)),
            "min_ms": float(s.min()),
            "max_ms": float(s.max())
        }

    kafka_to_received_stats = _lat_stats(pdf["kafka_to_received_ms"])
    published_to_received_stats = _lat_stats(pdf["published_to_received_ms"])

    # offsets per topic/partition ranges
    offset_ranges = {}
    for (t, p), g in pdf.groupby(["topic", "partition"]):
        try:
            offset_ranges.setdefault(t, {})[int(p)] = {
                "min_offset": int(g["offset"].min()),
                "max_offset": int(g["offset"].max()),
                "count": int(len(g))
            }
        except Exception:
            pass

    # produce a small summary dictionary
    metrics = {
        "generated_at": datetime.now(timezone.utc).isoformat(),
        "total_messages": total_msgs,
        "consumption_elapsed_seconds": elapsed,
        "throughput_msgs_per_sec": throughput,
        "per_topic": per_topic,
        "message_size": size_stats,
        "kafka_to_received_latency_ms": kafka_to_received_stats,
        "published_to_received_latency_ms": published_to_received_stats,
        "offset_ranges": offset_ranges,
    }

    # print a readable summary
    print("\nPIPELINE METRICS SUMMARY")
    print("========================")
    print(f"Total messages: {total_msgs}")
    if elapsed is not None:
        print(f"Consumption elapsed (s): {elapsed:.3f}")
    if throughput is not None:
        print(f"Throughput (msgs/sec): {throughput:.3f}")
    print(f"Message size (mean/p50/p90 bytes): {size_stats['mean_bytes']:.1f} / {size_stats['p50_bytes']} / {size_stats['p90_bytes']}")
    if kafka_to_received_stats:
        print(f"Kafka->received latency (mean/p90/p99 ms): {kafka_to_received_stats['mean_ms']:.1f} / {kafka_to_received_stats['p90_ms']:.1f} / {kafka_to_received_stats['p99_ms']:.1f}")
    else:
        print("Kafka->received latency: no timestamps available")
    if published_to_received_stats:
        print(f"Published->received (end-to-end) (median/p95 ms): {published_to_received_stats['median_ms']:.1f} / {published_to_received_stats['p95_ms']:.1f}")
    else:
        print("Published->received latency: no published timestamps available")
    print("Offsets by topic/partition:")
    for t, parts in offset_ranges.items():
        for p, info in parts.items():
            print(f"  {t} / partition {p}: offsets {info['min_offset']}..{info['max_offset']} ({info['count']} messages)")

    # optional save
    if SAVE_METRICS_JSON:
        try:
            METRICS_OUT.parent.mkdir(parents=True, exist_ok=True)
            with open(METRICS_OUT, "w", encoding="utf-8") as fo:
                json.dump(metrics, fo, indent=2, default=str)
            print(f"Metrics saved to {METRICS_OUT}")
        except Exception as e:
            print("Failed to save metrics:", e)

    return metrics

def _to_utc_iso(dt):
    """
    Convert various datetime-like inputs to an ISO8601 string with UTC tz,
    or return None if input is missing/invalid.
    """
    if dt is None or (isinstance(dt, float) and math.isnan(dt)):
        return None
    try:
        # Let pandas normalize many inputs (str, pd.Timestamp, datetime, numpy types)
        ts = pd.to_datetime(dt, utc=True, errors="coerce")
        if pd.isna(ts):
            return None
        # ts is tz-aware (UTC). Return ISO string with offset (e.g. 2025-11-27T08:00:00+00:00)
        return ts.isoformat()
    except Exception:
        return None

def insert_data_to_db_spark(spark_df, db_end_point: str, db_token: str, chunk_size: int = 500):
    """
    Insert rows from a Spark DataFrame into Astra DB (DataAPI).
    Ensures published_at is timezone-aware ISO strings (UTC) to avoid Serdes errors.
    Uses chunked insert_many to avoid very large single requests.

    Args:
        spark_df: Spark DataFrame containing at least these columns:
                  stock_symbol, summary, published_ts, url, sentiment_score
        db_end_point: Astra DB endpoint URL
        db_token: Astra DB token
        chunk_size: number of rows per insert_many call
    """
    # select the columns we care about (ignore missing columns gracefully)
    cols = []
    for c in ("stock_symbol", "summary", "published_ts", "url", "sentiment_score"):
        if c in spark_df.columns:
            cols.append(c)
        else:
            # if missing, create a literal column of None so pandas has the column
            spark_df = spark_df.withColumn(c, spark_df[c] * 0) if False else spark_df  # noop to keep lint happy

    pdf = spark_df.select("stock_symbol", "summary", "published_ts", "url", "sentiment_score").toPandas()

    if pdf.empty:
        print("No records to insert.")
        return

    # Build rows while normalizing published_at to tz-aware ISO strings
    rows = []
    for idx in range(len(pdf)):
        symbol = pdf.loc[idx, "stock_symbol"] if "stock_symbol" in pdf.columns else None
        summary = pdf.loc[idx, "summary"] if "summary" in pdf.columns else None
        pub_date = pdf.loc[idx, "published_ts"] if "published_ts" in pdf.columns else None
        news_url = pdf.loc[idx, "url"] if "url" in pdf.columns else None
        sentiment_score = pdf.loc[idx, "sentiment_score"] if "sentiment_score" in pdf.columns else None

        # convert published_ts to UTC ISO string (or None)
        published_at_iso = _to_utc_iso(pub_date)

        # Safe cast sentiment
        try:
            sentiment_score_int = int(sentiment_score) if sentiment_score is not None and not (isinstance(sentiment_score, float) and math.isnan(sentiment_score)) else None
        except Exception:
            sentiment_score_int = None

        row = {
            "stock_symbol": None if pd.isna(symbol) else symbol,
            "published_at": published_at_iso,
            "news_url": None if pd.isna(news_url) else news_url,
            "news_summary": None if pd.isna(summary) else summary,
            "sentiment_score": sentiment_score_int
        }
        rows.append(row)

    # init client and table
    client_db = DataAPIClient()
    database = client_db.get_database(db_end_point, token=db_token)
    table = database.get_table("news_sentiment")

    print(f"Prepared {len(rows)} rows. Inserting in chunks of {chunk_size}...")

    for i in range(0, len(rows), chunk_size):
        chunk = rows[i:i + chunk_size]
        try:
            table.insert_many(chunk)
            print(f"Inserted rows {i}..{i+len(chunk)-1}")
        except Exception as e:
            print(f"Failed to insert chunk starting at {i}: {e}")
            # optionally: raise or continue
            raise

    print("All done.")

# ----- MAIN -----
if __name__ == "__main__":
    spark = SparkSession.builder.appName("KafkaConsumerToSpark").getOrCreate()
    start_time1 = datetime.now()
    df = consume_one_consumer_per_topic()
    end_time1 = datetime.now()
    elapsed1 = (end_time1 - start_time1).total_seconds()
    print("Elapsed seconds for consumption:", elapsed1)

    # compute and print metrics
    metrics = compute_pipeline_metrics(df, consumption_elapsed_seconds=elapsed1)

    # optionally save Spark DF to CSV for inspection
    if SAVE_CSV:
        try:
            OUT.parent.mkdir(parents=True, exist_ok=True)
            df.toPandas().to_csv(OUT, index=False)
            print(f"Wrote messages CSV to {OUT}")
        except Exception as e:
            print("Failed to write CSV:", e)

    start_time2 = datetime.now()
    # Insert to DB (your original credentials; keep secure in production)
    db_end_point = 'https://d1091311-1ba8-4970-bd74-937d6cda55b0-us-east-2.apps.astra.datastax.com'
    db_token = 'AstraCS:YOUR_TOKEN_HERE'
    insert_data_to_db_spark(df, db_end_point, db_token)
    end_time2 = datetime.now()
    elapsed2 = (end_time2 - start_time2).total_seconds()
    print("Elapsed seconds for DB insert:", elapsed2)
    print("Elapsed seconds for consumption:", elapsed1)

In [ ]:
#Helper function to extract score from text
def extract_score_from_text(text: str) -> int:
    if not text:
        return 1
    m = re.search(r"(?m)^\s*([1-5])\s*$", text)
    if m:
        return int(m.group(1))
    m_all = re.findall(r"(?<!\d)([1-5])(?!\d)", text)
    if m_all:
        return int(m_all[-1])
    return 1

# Function to get stock sentiments using Groq API
def get_stock_sentiments(stock_ticker: str, news_summary: str,
                         groq_api_key: str,
                         model: str = "qwen/qwen3-32b",
                         max_retries: int = 3,
                         backoff_base: float = 1.0,
                         semaphore: BoundedSemaphore = None) -> int:
    """
    Synchronously call Groq and return a 1-5 integer sentiment.
    Optionally uses a semaphore to limit concurrency (pass same semaphore from caller).
    """
    # Acquire semaphore if provided
    if semaphore is not None:
        semaphore.acquire()

    try:
        sentiment_template = '''
Role:
You are a financial analyst specializing in interpreting news sentiment for stocks. 
Do NOT show your reasoning. Do NOT use "thinking" or any hidden tags.

Task:
Analyze the sentiment of the news summary toward the specified stock.

Input:
Stock Ticker: {stock_ticker}
News Summary: {news_summary}

Scoring Rules (integer only):
1 - Negative: Clearly unfavorable news; likely negative impact on the stock.
2 - Somewhat Negative: Slightly unfavorable; minor concerns.
3 - Neutral: Balanced or unclear impact.
4 - Somewhat Positive: Slightly favorable; minor optimism.
5 - Positive: Clearly favorable; likely positive impact on the stock.

Output:
Return exactly ONE CHARACTER: a single digit 1,2,3,4 or 5 and nothing else.
Do not include spaces, newline, punctuation, tags, labels, or any other characters.
'''
        prompt = sentiment_template.format(stock_ticker=stock_ticker, news_summary=news_summary)

        attempt = 0
        while attempt < max_retries:
            try:
                # instantiate client per-call for thread-safety
                client_groq = Groq(api_key=groq_api_key)

                response = client_groq.chat.completions.create(
                    model=model,
                    messages=[{"role": "user", "content": prompt}],
                    temperature=0
                )

                raw = ""
                try:
                    raw = response.choices[0].message.content
                except Exception:
                    raw = getattr(response.choices[0], "text", "") or str(response)
                raw = (raw or "").strip()
                score = extract_score_from_text(raw)
                return score

            except Exception as e:
                attempt += 1
                wait = backoff_base * (2 ** (attempt - 1))
                # jitter using random.random()
                jitter = 0.5 + 0.5 * random.random()
                wait = wait * jitter
                print(f"[Sentiment] attempt {attempt}/{max_retries} failed for {stock_ticker}: {e}. sleeping {wait:.1f}s")
                time.sleep(wait)

        # fallback on repeated failure
        print("[Sentiment] model calls exhausted; returning default 1 for", stock_ticker)
        return 1

    finally:
        if semaphore is not None:
            try:
                semaphore.release()
            except Exception:
                pass

#Function to insert data to DB in parallel after calculating sentiment score
def insert_data_to_db_parallel(input_df: pd.DataFrame,
                               groq_api_key: str,
                               db_end_point: str,
                               db_token: str,
                               batch_size: int = 500,
                               max_workers: int = 8,
                               max_concurrent_requests: int = 4,
                               sentiment_model: str = "qwen/qwen3-32b"):
    """
    Parallelized sentiment fetch using ThreadPoolExecutor + semaphore to bound concurrency.
    After sentiment fetch, inserts batched rows via astrapy.
    """
    # create DB client once
    client_db = DataAPIClient()
    database = client_db.get_database(db_end_point, token=db_token)
    table = database.get_table("news_sentiment")

    n = len(input_df)
    rows_to_insert = []

    # semaphore limits in-flight Groq calls
    semaphore = BoundedSemaphore(max_concurrent_requests)

    with ThreadPoolExecutor(max_workers=max_workers) as executor:
        # a mapping from future -> row index (so we can match responses)
        future_to_idx = {}

        # submit sentiment tasks for each row (or for a chunk — here per-row)
        for idx in range(n):
            symbol = input_df.Stock_symbol.iloc[idx]
            summary = input_df.Textrank_summary.iloc[idx]
            pub_date = input_df.Date_new.iloc[idx]

            # ensure valid datetime
            try:
                start = pd.to_datetime(pub_date)
            except Exception:
                print(f"Skipping row {idx}: invalid Date_new {pub_date}")
                continue

            # submit the sentiment work to thread pool
            fut = executor.submit(get_stock_sentiments,
                                  symbol,
                                  summary,
                                  groq_api_key,
                                  sentiment_model,
                                  3,      # retries inside
                                  1.0,    # base backoff
                                  semaphore)
            future_to_idx[fut] = idx

        # As futures complete, build rows and insert in batches
        processed = 0
        rows_buffer = []
        for fut in as_completed(future_to_idx):
            idx = future_to_idx[fut]
            try:
                score = fut.result()  # will block until available or raise
            except Exception as e:
                print(f"Sentiment future failed for idx {idx}: {e}")
                score = 1

            # now build the DB row for this index
            symbol = input_df.Stock_symbol.iloc[idx]
            summary = input_df.Textrank_summary.iloc[idx]
            pub_date = input_df.Date_new.iloc[idx]
            news_url = input_df.Url.iloc[idx] if "Url" in input_df.columns else None

            try:
                start = pd.to_datetime(pub_date)
                published_at = start.to_pydatetime()
            except Exception:
                published_at = None

            row = {
                "stock_symbol": symbol,
                "published_at": published_at,
                "news_url": news_url,
                "news_summary": summary,
                "sentiment_score": int(score)
            }
            rows_buffer.append(row)
            processed += 1

            # insert in batches
            if len(rows_buffer) >= batch_size:
                table.insert_many(rows_buffer)
                print(f"Inserted batch of {len(rows_buffer)} rows (processed {processed}/{n})")
                rows_buffer = []

            # progress log
            if processed % 20 == 0 or processed == n:
                print(f"Prepared+sent {processed}/{n} rows")

        # final flush
        if rows_buffer:
            table.insert_many(rows_buffer)
            print(f"Inserted final {len(rows_buffer)} rows")

    print("All done.")

In [ ]:
#Kafka consumer to read messages and load to Astra DB after evaluating the sentiment score
BOOTSTRAP = "localhost:9092"
TOPICS = ["NASDAQ_AMGN"]
GROUP = "one-shot-test-group"
TIMEOUT_SECONDS = 10
OUT = Path("out/one_shot.csv")

conf = {
    "bootstrap.servers": BOOTSTRAP,
    "group.id": GROUP,
    "auto.offset.reset": "earliest",  # read existing messages for a fresh group
    "enable.auto.commit": False
}

def parse_value(b):
    if b is None: return None
    try:
        s = b.decode("utf-8") if isinstance(b, bytes) else str(b)
        return json.loads(s)
    except:
        return s

c = Consumer(conf)
c.subscribe(TOPICS)
print("Subscribed:", TOPICS)
records = []
start = datetime.now()
while (datetime.now() - start).seconds < TIMEOUT_SECONDS:
    msg = c.poll(timeout=1.0)
    if msg is None:
        continue
    if msg.error():
        if msg.error().code() == KafkaError._PARTITION_EOF:
            continue
        print("Error:", msg.error())
        continue
    rec = {
        "topic": msg.topic(),
        "partition": msg.partition(),
        "offset": msg.offset(),
        "timestamp_ms": msg.timestamp()[1],
        "timestamp": datetime.now().isoformat(),
        "key": (msg.key().decode("utf-8") if isinstance(msg.key(), bytes) else msg.key()),
        "value": parse_value(msg.value())
    }
    #print("Got:", rec)
    records.append(rec)

c.close()

'''
if records:
    df = pd.DataFrame(records)
    OUT.parent.mkdir(parents=True, exist_ok=True)
    header = not OUT.exists()
    #df.to_csv(OUT, mode="a", header=header, index=False)
    print(f"Wrote {len(records)} records to {OUT.resolve()}")
else:
    print("No messages received in timeout window.")

'''

if records:
    df = pd.DataFrame(records)
    df['stock_symbol'] = df['value'].apply(lambda x: x.get('stock_symbol') if isinstance(x, dict) else None)
    df['summary'] = df['value'].apply(lambda x: x.get('summary') if isinstance(x, dict) else None)
    df['published_date'] = df['value'].apply(lambda x: x.get('event_time') if isinstance(x, dict) else None)
    df['date_new'] = pd.to_datetime(df.published_date, utc=True)
    df['url'] = df['value'].apply(lambda x: x.get('url') if isinstance(x, dict) else None)
    
    insert_data_to_db_parallel(
                                input_df=df,
                                groq_api_key=grok_api_key,
                                db_end_point=db_end_point,
                                db_token=db_token,
                                batch_size=500,
                                max_workers=16,               # threadpool size
                                max_concurrent_requests=4,    # concurrency cap to the Groq API
                                sentiment_model="qwen/qwen3-32b"
)
    
    print(f"Wrote {len(records)} records to {OUT.resolve()}")
else:
    print("No messages received in timeout window.")


In [ ]:
#Function to insert data to DB in parallel with pre-calculated sentiment score
def insert_data_to_db(input_df: pd.DataFrame,
                               db_end_point: str,
                               db_token: str
                    ):

    # create DB client once
    client_db = DataAPIClient()
    database = client_db.get_database(db_end_point, token=db_token)
    table = database.get_table("news_sentiment")

    n = len(input_df)
    rows_to_insert = []

    for idx in range(n):
        symbol = input_df.Stock_symbol.iloc[idx]
        summary = input_df.Textrank_summary.iloc[idx]
        pub_date = input_df.Date_new.iloc[idx]
        news_url = input_df.Url.iloc[idx]
        sentiment_score = input_df.sentiment_score.iloc[idx]

        processed = 0
        rows_buffer = []

        row = {
                "stock_symbol": symbol,
                "published_at": pub_date,
                "news_url": news_url,
                "news_summary": summary,
                "sentiment_score": int(sentiment_score)
            }
        rows_buffer.append(row)
        processed += 1

        # insert in batches
        if len(rows_buffer) >= batch_size:
            table.insert_many(rows_buffer)
            print(f"Inserted batch of {len(rows_buffer)} rows (processed {processed}/{n})")
            rows_buffer = []

        # progress log
        if processed % 20 == 0 or processed == n:
            print(f"Prepared+sent {processed}/{n} rows")

        # final flush
        if rows_buffer:
            table.insert_many(rows_buffer)
            print(f"Inserted final {len(rows_buffer)} rows")

    print("All done.")